In [1]:
%pip install -q gradio plotly

In [2]:

import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px

# --------------------------------------------------
# 1. PROJECT SETTINGS
# --------------------------------------------------

TITLE = "Marketing-to-Revenue Attribution"

GREEN = "#24483F"
GOLD = "#C49A52"
SAGE = "#91AA9A"
DARK = "#263C34"
IVORY = "#F7F5EF"
PALE = "#E4E9E2"

np.random.seed(42)

# --------------------------------------------------
# 2. CREATE SAMPLE MARKETING DATA
# --------------------------------------------------

channels = {
    "Social Media": {
        "campaigns": ["Brand Awareness", "Product Campaign", "Retargeting"],
        "cost_per_lead": 42,
        "qualification_rate": 0.36,
        "conversion_rate": 0.19,
        "revenue": 2600
    },
    "Email Marketing": {
        "campaigns": ["Welcome Series", "Lead Nurturing", "Product Updates"],
        "cost_per_lead": 18,
        "qualification_rate": 0.52,
        "conversion_rate": 0.31,
        "revenue": 3200
    },
    "Paid Search": {
        "campaigns": ["High Intent Search", "Product Keywords", "Competitor Search"],
        "cost_per_lead": 65,
        "qualification_rate": 0.58,
        "conversion_rate": 0.34,
        "revenue": 3900
    },
    "Webinars": {
        "campaigns": ["Industry Webinar", "Product Workshop", "Expert Session"],
        "cost_per_lead": 55,
        "qualification_rate": 0.64,
        "conversion_rate": 0.38,
        "revenue": 4500
    },
    "Content Marketing": {
        "campaigns": ["Blog Campaign", "Case Study", "Downloadable Guide"],
        "cost_per_lead": 28,
        "qualification_rate": 0.45,
        "conversion_rate": 0.25,
        "revenue": 3000
    }
}

rows = []

for i in range(1200):
    channel = np.random.choice(list(channels.keys()))
    details = channels[channel]
    campaign = np.random.choice(details["campaigns"])

    lead_score = int(np.clip(np.random.normal(65, 18), 10, 100))

    qualified_probability = np.clip(
        details["qualification_rate"] + (lead_score - 65) * 0.003,
        0.05, 0.90
    )
    qualified = np.random.random() < qualified_probability

    if qualified:
        conversion_probability = np.clip(
            details["conversion_rate"] + (lead_score - 65) * 0.002,
            0.05, 0.85
        )
        converted = np.random.random() < conversion_probability
    else:
        converted = False

    lead_cost = details["cost_per_lead"] * np.random.uniform(0.8, 1.2)
    revenue = (
        details["revenue"] * np.random.uniform(0.75, 1.25)
        if converted else 0
    )

    rows.append({
        "Lead ID": f"LEAD-{i+1:04d}",
        "Channel": channel,
        "Campaign": campaign,
        "Lead Score": lead_score,
        "Qualified": "Yes" if qualified else "No",
        "Converted": "Yes" if converted else "No",
        "Marketing Cost": round(lead_cost, 2),
        "Revenue": round(revenue, 2),
        "Days to Convert": (
            int(np.random.randint(3, 46)) if converted else None
        )
    })

df = pd.DataFrame(rows)

# --------------------------------------------------
# 3. PREPARE ATTRIBUTION METRICS
# --------------------------------------------------

def build_channel_summary(data):
    summary = data.groupby("Channel").agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Marketing_Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()

    summary["Lead-to-Qualified %"] = (
        summary["Qualified_Leads"] / summary["Leads"] * 100
    ).round(1)

    summary["Lead-to-Conversion %"] = (
        summary["Conversions"] / summary["Leads"] * 100
    ).round(1)

    summary["Cost per Lead"] = (
        summary["Marketing_Spend"] / summary["Leads"]
    ).round(2)

    summary["Revenue per Conversion"] = (
        summary["Revenue"] / summary["Conversions"].replace(0, np.nan)
    ).fillna(0).round(2)

    summary["ROAS"] = (
        summary["Revenue"] / summary["Marketing_Spend"].replace(0, np.nan)
    ).fillna(0).round(2)

    return summary

summary = build_channel_summary(df)

total_leads = len(df)
total_qualified = (df["Qualified"] == "Yes").sum()
total_conversions = (df["Converted"] == "Yes").sum()
total_spend = df["Marketing Cost"].sum()
total_revenue = df["Revenue"].sum()
overall_conversion = total_conversions / total_leads * 100
overall_roas = total_revenue / total_spend if total_spend else 0

# --------------------------------------------------
# 4. CREATE CHARTS
# --------------------------------------------------

def create_funnel():
    funnel = pd.DataFrame({
        "Stage": ["Leads Generated", "Qualified Leads", "Conversions"],
        "Count": [total_leads, total_qualified, total_conversions]
    })

    fig = px.funnel(
        funnel,
        x="Count",
        y="Stage",
        title="Marketing Funnel: From Leads to Revenue",
        color="Stage",
        color_discrete_sequence=[GREEN, SAGE, GOLD]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False
    )
    return fig

def create_revenue_chart():
    chart_data = summary.sort_values("Revenue", ascending=True)

    fig = px.bar(
        chart_data,
        x="Revenue",
        y="Channel",
        orientation="h",
        title="Revenue Attributed to Each Channel",
        color="Revenue",
        color_continuous_scale=[SAGE, GREEN],
        labels={"Revenue": "Revenue", "Channel": "Marketing Channel"}
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        coloraxis_showscale=False
    )
    return fig

def create_conversion_chart():
    chart_data = summary.sort_values("Lead-to-Conversion %", ascending=False)

    fig = px.bar(
        chart_data,
        x="Channel",
        y="Lead-to-Conversion %",
        title="Lead-to-Customer Conversion Rate by Channel",
        color="Channel",
        color_discrete_sequence=[GREEN, GOLD, SAGE, "#748A80", "#D8C9A7"]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False,
        xaxis_title="Marketing Channel",
        yaxis_title="Conversion Rate (%)"
    )
    return fig

# --------------------------------------------------
# 5. INTERACTIVE FUNCTIONS
# --------------------------------------------------

def get_campaign_data(selected_channel):
    data = df.copy()

    if selected_channel != "All Channels":
        data = data[data["Channel"] == selected_channel]

    campaign_summary = data.groupby(["Channel", "Campaign"]).agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()

    campaign_summary["Conversion Rate (%)"] = (
        campaign_summary["Conversions"] / campaign_summary["Leads"] * 100
    ).round(1)

    campaign_summary["ROAS"] = (
        campaign_summary["Revenue"] /
        campaign_summary["Spend"].replace(0, np.nan)
    ).fillna(0).round(2)

    campaign_summary["Spend"] = campaign_summary["Spend"].round(2)
    campaign_summary["Revenue"] = campaign_summary["Revenue"].round(2)

    return campaign_summary

def explore_channel(selected_channel):
    channel_data = summary[summary["Channel"] == selected_channel].iloc[0]

    explanation = f"""
## {selected_channel}

- **Leads generated:** {int(channel_data['Leads'])}
- **Qualified leads:** {int(channel_data['Qualified_Leads'])}
- **Conversions:** {int(channel_data['Conversions'])}
- **Lead-to-conversion rate:** {channel_data['Lead-to-Conversion %']}%
- **Marketing spend:** ${channel_data['Marketing_Spend']:,.2f}
- **Attributed revenue:** ${channel_data['Revenue']:,.2f}
- **Return on ad spend (ROAS):** {channel_data['ROAS']:.2f}x

**How to interpret this:** Compare the channel's conversion rate, revenue
and ROAS together. High lead volume alone does not guarantee strong revenue.
"""
    return explanation

def generate_insights():
    best_roas = summary.loc[summary["ROAS"].idxmax()]
    best_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmax()]
    lowest_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmin()]
    highest_spend = summary.loc[summary["Marketing_Spend"].idxmax()]

    insights = f"""
# Marketing Performance Insights

### 1. Strongest revenue return
**{best_roas['Channel']}** has the highest ROAS at **{best_roas['ROAS']:.2f}x**.
Review its audience, campaign message and lead quality to identify what may
be worth repeating.

### 2. Best lead-to-customer conversion
**{best_conversion['Channel']}** has the highest conversion rate at
**{best_conversion['Lead-to-Conversion %']}%**. Consider whether similar
follow-up practices could work for other channels.

### 3. Funnel improvement opportunity
**{lowest_conversion['Channel']}** has the lowest conversion rate at
**{lowest_conversion['Lead-to-Conversion %']}%**. Investigate lead quality,
landing-page experience and sales follow-up before reducing its budget.

### 4. Spending review
**{highest_spend['Channel']}** has the highest marketing spend
(${highest_spend['Marketing_Spend']:,.2f}). Compare that investment with its
attributed revenue and ROAS before deciding whether to reallocate funds.

### Recommended next steps
- Review channels with many leads but relatively few conversions.
- Compare qualified-lead rates and conversion rates.
- Test changes to targeting, messaging and follow-up.
- Reassess performance using real campaign and sales data.

*These are data-based rules and recommendations, not predictions from a
trained generative AI model.*
"""
    return insights

# --------------------------------------------------
# 6. DASHBOARD DESIGN
# --------------------------------------------------

css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    background: #F7F5EF !important;
    color: #263C34 !important;
    font-family: Arial, sans-serif !important;
}
#main-header {
    padding: 28px 30px;
    border-radius: 8px;
    background: #24483F;
    color: #FFFFFF !important;
    margin-bottom: 22px;
}
#main-header h1 {
    color: #FFFFFF !important;
    font-size: 29px !important;
    font-weight: 600 !important;
}
#main-header p {
    color: #E4E9E2 !important;
    font-size: 15px !important;
}
.gradio-container h2,
.gradio-container h3 {
    color: #24483F !important;
}
.gradio-container label {
    color: #263C34 !important;
}
.gradio-container button {
    border-radius: 6px !important;
}
.gradio-container input {
    background: #FFFFFF !important;
    color: #263C34 !important;
    border-color: #D8DED4 !important;
}
.gradio-container .block {
    border-color: #E0E1D8 !important;
    box-shadow: none !important;
}
"""

theme = gr.themes.Base(
    primary_hue=gr.themes.Color(
        c50="#F0F4EF",
        c100="#DCE7DC",
        c200="#C0D2C2",
        c300="#A2BDA8",
        c400="#759A81",
        c500="#527A61",
        c600="#3D654E",
        c700="#315440",
        c800="#24483F",
        c900="#1C3932",
        c950="#132923"
    ),
    neutral_hue="stone",
    font=["Arial", "sans-serif"]
)

# --------------------------------------------------
# 7. BUILD THE INTERACTIVE APP
# --------------------------------------------------

with gr.Blocks(theme=theme, css=css, title=TITLE) as demo:

    gr.HTML("""
    <div id="main-header">
        <h1>Marketing-to-Revenue Attribution</h1>
        <p>From campaign exposure to qualified leads, conversions, and revenue.</p>
    </div>
    """)

    gr.Markdown(
        "Explore where marketing generates value, where the funnel loses "
        "potential customers, and which channels deserve closer review."
    )

    with gr.Tabs():

        # TAB 1: EXECUTIVE OVERVIEW
        with gr.Tab("Executive Overview"):
            gr.Markdown("## Business Performance at a Glance")

            with gr.Row():
                gr.Number(
                    value=total_leads,
                    label="Total Leads",
                    precision=0,
                    interactive=False
                )
                gr.Number(
                    value=total_qualified,
                    label="Qualified Leads",
                    precision=0,
                    interactive=False
                )
                gr.Number(
                    value=total_conversions,
                    label="Conversions",
                    precision=0,
                    interactive=False
                )

            with gr.Row():
                gr.Number(
                    value=round(total_spend, 2),
                    label="Marketing Spend ($)",
                    precision=2,
                    interactive=False
                )
                gr.Number(
                    value=round(total_revenue, 2),
                    label="Attributed Revenue ($)",
                    precision=2,
                    interactive=False
                )
                gr.Number(
                    value=round(overall_roas, 2),
                    label="Overall ROAS (x)",
                    precision=2,
                    interactive=False
                )

            with gr.Row():
                gr.Plot(value=create_funnel())
                gr.Plot(value=create_revenue_chart())

            gr.Markdown(
                f"**Overall lead-to-customer conversion rate:** "
                f"{overall_conversion:.1f}%"
            )

        # TAB 2: FUNNEL ANALYSIS
        with gr.Tab("Funnel Analysis"):
            gr.Markdown("## Where Are Potential Customers Lost?")
            gr.Markdown(
                "Follow the journey from lead generation to qualification "
                "and conversion. A lower count at each stage represents "
                "potential drop-off."
            )

            gr.Plot(value=create_funnel())
            gr.Plot(value=create_conversion_chart())

            funnel_table = pd.DataFrame({
                "Funnel Stage": [
                    "Leads Generated",
                    "Qualified Leads",
                    "Conversions"
                ],
                "Number of Leads": [
                    total_leads,
                    total_qualified,
                    total_conversions
                ],
                "Share of Original Leads (%)": [
                    100,
                    round(total_qualified / total_leads * 100, 1),
                    round(total_conversions / total_leads * 100, 1)
                ]
            })

            gr.Dataframe(
                value=funnel_table,
                label="Funnel Stage Summary",
                interactive=False
            )

        # TAB 3: CAMPAIGN PERFORMANCE
        with gr.Tab("Campaign Performance"):
            gr.Markdown("## Compare Individual Campaigns")

            channel_filter = gr.Dropdown(
                choices=["All Channels"] + list(channels.keys()),
                value="All Channels",
                label="Filter by Marketing Channel"
            )

            campaign_table = gr.Dataframe(
                value=get_campaign_data("All Channels"),
                label="Campaign Performance",
                interactive=False
            )

            channel_filter.change(
                fn=get_campaign_data,
                inputs=channel_filter,
                outputs=campaign_table
            )

            gr.Markdown(
                "**ROAS** means revenue attributed to a channel divided by "
                "its marketing spend. A higher ROAS is not the same as "
                "higher total profit."
            )

        # TAB 4: ATTRIBUTION EXPLORER
        with gr.Tab("Attribution Explorer"):
            gr.Markdown("## Explore a Marketing Channel")
            gr.Markdown(
                "Select a channel to examine its lead generation, "
                "conversion and revenue contribution."
            )

            channel_select = gr.Dropdown(
                choices=list(channels.keys()),
                value="Paid Search",
                label="Select Marketing Channel"
            )

            channel_report = gr.Markdown(
                value=explore_channel("Paid Search")
            )

            channel_select.change(
                fn=explore_channel,
                inputs=channel_select,
                outputs=channel_report
            )

        # TAB 5: AI-STYLE INSIGHTS
        with gr.Tab("Marketing Insights"):
            gr.Markdown("## Recommendations for Marketing Decisions")

            insight_output = gr.Markdown(value=generate_insights())

            refresh_button = gr.Button(
                "Refresh Insights",
                variant="primary"
            )

            refresh_button.click(
                fn=generate_insights,
                inputs=[],
                outputs=insight_output
            )

            gr.Markdown(
                "Recommendations are generated from the sample data using "
                "predefined business rules. Validate them against real "
                "campaign results before making budget decisions."
            )

    gr.Markdown(
        "---\n"
        "**Project note:** This dashboard uses synthetic demonstration "
        "data. Channel attribution is illustrative and assigns each lead "
        "to one recorded channel; it does not track a real customer's "
        "complete multi-touch journey."
    )

# --------------------------------------------------
# 8. LAUNCH THE DASHBOARD
# --------------------------------------------------

demo.launch(share=True, debug=False)


/tmp/ipykernel_504/1699397452.py:384: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=theme, css=css, title=TITLE) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://608741e617ab9f6d7a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [4]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px

# ------------------------------------------------ --
# 1. PROJECT SETTINGS
# ------------------------------------------------ --
TITLE = "Marketing-to-Revenue Attribution"
GREEN = "#24483F"
GOLD = "#C49A52"
SAGE = "#91AA9A"
DARK = "#263C34"
IVORY = "#F7F5EF"
PALE = "#E4E9E2"

np.random.seed(42)

# ------------------------------------------------ --
# 2. CREATE SAMPLE MARKETING DATA
# ------------------------------------------------ --
channels = {
    "Social Media": {
        "campaigns": ["Brand Awareness", "Product Campaign", "Retargeting"],
        "cost_per_lead": 42,
        "qualification_rate": 0.36,
        "conversion_rate": 0.19,
        "revenue": 2600,
    },
    "Email Marketing": {
        "campaigns": ["Welcome Series", "Lead Nurturing", "Product Updates"],
        "cost_per_lead": 18,
        "qualification_rate": 0.52,
        "conversion_rate": 0.31,
        "revenue": 3200,
    },
    "Paid Search": {
        "campaigns": ["High Intent Search", "Product Keywords", "Competitor Search"],
        "cost_per_lead": 65,
        "qualification_rate": 0.58,
        "conversion_rate": 0.34,
        "revenue": 3900,
    },
    "Webinars": {
        "campaigns": ["Industry Webinar", "Product Workshop", "Expert Session"],
        "cost_per_lead": 55,
        "qualification_rate": 0.64,
        "conversion_rate": 0.38,
        "revenue": 4500,
    },
    "Content Marketing": {
        "campaigns": ["Blog Campaign", "Case Study", "Downloadable Guide"],
        "cost_per_lead": 28,
        "qualification_rate": 0.45,
        "conversion_rate": 0.25,
        "revenue": 3000,
    },
}

rows = []
for i in range(1200):
    channel = np.random.choice(list(channels.keys()))
    details = channels[channel]
    campaign = np.random.choice(details["campaigns"])
    lead_score = int(np.clip(np.random.normal(65, 18), 10, 100))
    qualified_probability = np.clip(
        details["qualification_rate"] + (lead_score - 65) * 0.003,
        0.05, 0.90
    )
    qualified = np.random.random() < qualified_probability
    if qualified:
        conversion_probability = np.clip(
            details["conversion_rate"] + (lead_score - 65) * 0.002,
            0.05, 0.85
        )
        converted = np.random.random() < conversion_probability
    else:
        converted = False
    lead_cost = details["cost_per_lead"] * np.random.uniform(0.8, 1.2)
    revenue = (
        details["revenue"] * np.random.uniform(0.75, 1.25)
        if converted else 0
    )
    rows.append({
        "Lead ID": f"LEAD-{i+1:04d}",
        "Channel": channel,
        "Campaign": campaign,
        "Lead Score": lead_score,
        "Qualified": "Yes" if qualified else "No",
        "Converted": "Yes" if converted else "No",
        "Marketing Cost": round(lead_cost, 2),
        "Revenue": round(revenue, 2),
        "Days to Convert": (
            int(np.random.randint(3, 46)) if converted else None
        )
    })

df = pd.DataFrame(rows)

# ------------------------------------------------ --
# 3. PREPARE ATTRIBUTION METRICS
# ------------------------------------------------ --
def build_channel_summary(data):
    summary = data.groupby("Channel").agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Marketing_Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()
    summary["Lead-to-Qualified %"] = (
        summary["Qualified_Leads"] / summary["Leads"] * 100
    ).round(1)
    summary["Lead-to-Conversion %"] = (
        summary["Conversions"] / summary["Leads"] * 100
    ).round(1)
    summary["Cost per Lead"] = (
        summary["Marketing_Spend"] / summary["Leads"]
    ).round(2)
    summary["Revenue per Conversion"] = (
        summary["Revenue"] / summary["Conversions"].replace(0, np.nan)
    ).fillna(0).round(2)
    summary["ROAS"] = (
        summary["Revenue"] / summary["Marketing_Spend"].replace(0, np.nan)
    ).fillna(0).round(2)
    return summary

summary = build_channel_summary(df)
total_leads = len(df)
total_qualified = (df["Qualified"] == "Yes").sum()
total_conversions = (df["Converted"] == "Yes").sum()
total_spend = df["Marketing Cost"].sum()
total_revenue = df["Revenue"].sum()
overall_conversion = total_conversions / total_leads * 100
overall_roas = total_revenue / total_spend if total_spend else 0

# ------------------------------------------------ --
# 4. CREATE CHARTS
# ------------------------------------------------ --
def create_funnel():
    funnel = pd.DataFrame({
        "Stage": ["Leads Generated", "Qualified Leads", "Conversions"],
        "Count": [total_leads, total_qualified, total_conversions]
    })
    fig = px.funnel(
        funnel,
        x="Count",
        y="Stage",
        title="Marketing Funnel: From Leads to Revenue",
        color="Stage",
        color_discrete_sequence=[GREEN, SAGE, GOLD]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False
    )
    return fig

def create_revenue_chart():
    chart_data = summary.sort_values("Revenue", ascending=True)
    fig = px.bar(
        chart_data,
        x="Revenue",
        y="Channel",
        orientation="h",
        title="Revenue Attributed to Each Channel",
        color="Revenue",
        color_continuous_scale=[SAGE, GREEN],
        labels={"Revenue": "Revenue", "Channel": "Marketing Channel"}
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        coloraxis_showscale=False
    )
    return fig

def create_conversion_chart():
    chart_data = summary.sort_values("Lead-to-Conversion %", ascending=False)
    fig = px.bar(
        chart_data,
        x="Channel",
        y="Lead-to-Conversion %",
        title="Lead-to-Customer Conversion Rate by Channel",
        color="Channel",
        color_discrete_sequence=[GREEN, GOLD, SAGE, "#748A80", "#D8C9A7"]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False,
        xaxis_title="Marketing Channel",
        yaxis_title="Conversion Rate (%)"
    )
    return fig

# ------------------------------------------------ --
# 5. INTERACTIVE FUNCTIONS
# ------------------------------------------------ --
def get_campaign_data(selected_channel):
    data = df.copy()
    if selected_channel != "All Channels":
        data = data[data["Channel"] == selected_channel]
    campaign_summary = data.groupby(["Channel", "Campaign"]).agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()
    campaign_summary["Conversion Rate (%)"] = (
        campaign_summary["Conversions"] / campaign_summary["Leads"] * 100
    ).round(1)
    campaign_summary["ROAS"] = (
        campaign_summary["Revenue"] /
        campaign_summary["Spend"].replace(0, np.nan)
    ).fillna(0).round(2)
    campaign_summary["Spend"] = campaign_summary["Spend"].round(2)
    campaign_summary["Revenue"] = campaign_summary["Revenue"].round(2)
    return campaign_summary

def explore_channel(selected_channel):
    channel_data = summary[summary["Channel"] == selected_channel].iloc[0]
    explanation = f"""## {selected_channel}
- **Leads generated:** {int(channel_data['Leads'])}
- **Qualified leads:** {int(channel_data['Qualified_Leads'])}
- **Conversions:** {int(channel_data['Conversions'])}
- **Lead-to-conversion rate:** {channel_data['Lead-to-Conversion %']}%
- **Marketing spend:** ${channel_data['Marketing_Spend']:,.2f}
- **Attributed revenue:** ${channel_data['Revenue']:,.2f}
- **Return on ad spend (ROAS):** {channel_data['ROAS']:.2f}x

**How to interpret this:** Compare the channel's conversion rate, revenue and ROAS together. High lead volume alone does not guarantee strong revenue."""
    return explanation

def generate_insights():
    best_roas = summary.loc[summary["ROAS"].idxmax()]
    best_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmax()]
    lowest_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmin()]
    highest_spend = summary.loc[summary["Marketing_Spend"].idxmax()]
    insights = f"""# Marketing Performance Insights

### 1. Strongest revenue return
**{best_roas['Channel']}** has the highest ROAS at **{best_roas['ROAS']:.2f}x**. Review its audience, campaign message and lead quality to identify what may be worth repeating.

### 2. Best lead-to-customer conversion
**{best_conversion['Channel']}** has the highest conversion rate at **{best_conversion['Lead-to-Conversion %']}%**. Consider whether similar follow-up practices could work for other channels.

### 3. Funnel improvement opportunity
**{lowest_conversion['Channel']}** has the lowest conversion rate at **{lowest_conversion['Lead-to-Conversion %']}%**. Investigate lead quality, landing-page experience and sales follow-up before reducing its budget.

### 4. Spending review
**{highest_spend['Channel']}** has the highest marketing spend (${highest_spend['Marketing_Spend']:,.2f}). Compare that investment with its attributed revenue and ROAS before deciding whether to reallocate funds.

### Recommended next steps
- Review channels with many leads but relatively few conversions.
- Compare qualified-lead rates and conversion rates.
- Test changes to targeting, messaging and follow-up.
- Reassess performance using real campaign and sales data.

*These are data-based rules and recommendations, not predictions from a trained generative AI model.*"""
    return insights
# ------------------------------------------------ --
# 6. DASHBOARD DESIGN & CONTRAST FIXES
# ------------------------------------------------ --
css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    background: #F7F5EF !important;
    color: #263C34 !important;
    font-family: Arial, sans-serif !important;
}
#main-header {
    padding: 28px 30px;
    border-radius: 8px;
    background: #24483F;
    color: #FFFFFF !important;
    margin-bottom: 22px;
}
#main-header h1 {
    color: #FFFFFF !important;
    font-size: 29px !important;
    font-weight: 600 !important;
}
#main-header p {
    color: #E4E9E2 !important;
    font-size: 15px !important;
}
.gradio-container h2, .gradio-container h3 {
    color: #24483F !important;
}
.gradio-container label {
    color: #263C34 !important;
    font-weight: 600 !important;
}
.gradio-container button {
    border-radius: 6px !important;
}

/* --- FORCE VISIBLE TEXT ON ALL NUMBER / KPI INPUTS --- */
.gradio-container input,
.gradio-container textarea,
.gradio-container select,
.gradio-container .number-wrap input,
.gradio-container input[type="number"],
.gradio-container .block.number input,
.gradio-container .wrap input {
    color: #263C34 !important;
    -webkit-text-fill-color: #263C34 !important;
    font-weight: bold !important;
    background-color: #FFFFFF !important;
    opacity: 1 !important;
}

/* Style all metric and number card containers explicitly */
.gradio-container div[data-testid="number"],
.gradio-container .number-wrap,
.gradio-container .block.number {
    background-color: #FFFFFF !important;
    border: 1px solid #D8DED4 !important;
    border-radius: 8px !important;
    padding: 10px !important;
    color: #263C34 !important;
}

.gradio-container .block {
    border-color: #E0E1D8 !important;
    box-shadow: none !important;
}
"""

# ------------------------------------------------ --
# 7. BUILD THE INTERACTIVE APP
# ------------------------------------------------ --
with gr.Blocks(theme=theme, css=css, title=TITLE) as demo:
    gr.HTML("""
    <div id="main-header">
        <h1>Marketing-to-Revenue Attribution</h1>
        <p>From campaign exposure to qualified leads, conversions, and revenue.</p>
    </div>
    """)
    gr.Markdown(
        " Explore where marketing generates value, where the funnel loses "
        " potential customers, and which channels deserve closer review. "
    )
    with gr.Tabs():
        # TAB 1: EXECUTIVE OVERVIEW
        with gr.Tab("Executive Overview"):
            gr.Markdown(" ## Business Performance at a Glance ")
            with gr.Row():
                gr.Number(
                    value=total_leads,
                    label="Total Leads",
                    precision=0,
                    interactive=False
                )
                gr.Number(
                    value=total_qualified,
                    label="Qualified Leads",
                    precision=0,
                    interactive=False
                )
                gr.Number(
                    value=total_conversions,
                    label="Conversions",
                    precision=0,
                    interactive=False
                )
            with gr.Row():
                gr.Number(
                    value=round(total_spend, 2),
                    label="Marketing Spend ($)",
                    precision=2,
                    interactive=False
                )
                gr.Number(
                    value=round(total_revenue, 2),
                    label="Attributed Revenue ($)",
                    precision=2,
                    interactive=False
                )
                gr.Number(
                    value=round(overall_roas, 2),
                    label="Overall ROAS (x)",
                    precision=2,
                    interactive=False
                )
            with gr.Row():
                gr.Plot(value=create_funnel())
                gr.Plot(value=create_revenue_chart())
            gr.Markdown(
                f" **Overall lead-to-customer conversion rate:** "
                f" {overall_conversion:.1f}% "
            )

        # TAB 2: FUNNEL ANALYSIS
        with gr.Tab("Funnel Analysis"):
            gr.Markdown(" ## Where Are Potential Customers Lost? ")
            gr.Markdown(
                " Follow the journey from lead generation to qualification "
                " and conversion. A lower count at each stage represents "
                " potential drop-off. "
            )
            gr.Plot(value=create_funnel())
            gr.Plot(value=create_conversion_chart())
            funnel_table = pd.DataFrame({
                "Funnel Stage": [
                    "Leads Generated",
                    "Qualified Leads",
                    "Conversions"
                ],
                "Number of Leads": [
                    total_leads,
                    total_qualified,
                    total_conversions
                ],
                "Share of Original Leads (%)": [
                    100,
                    round(total_qualified / total_leads * 100, 1),
                    round(total_conversions / total_leads * 100, 1)
                ]
            })
            gr.Dataframe(
                value=funnel_table,
                label="Funnel Stage Summary",
                interactive=False
            )

        # TAB 3: CAMPAIGN PERFORMANCE
        with gr.Tab("Campaign Performance"):
            gr.Markdown(" ## Compare Individual Campaigns ")
            channel_filter = gr.Dropdown(
                choices=["All Channels"] + list(channels.keys()),
                value="All Channels",
                label="Filter by Marketing Channel"
            )
            campaign_table = gr.Dataframe(
                value=get_campaign_data("All Channels"),
                label="Campaign Performance",
                interactive=False
            )
            channel_filter.change(
                fn=get_campaign_data,
                inputs=channel_filter,
                outputs=campaign_table
            )
            gr.Markdown(
                " **ROAS** means revenue attributed to a channel divided by "
                " its marketing spend. A higher ROAS is not the same as "
                " higher total profit. "
            )

        # TAB 4: ATTRIBUTION EXPLORER
        with gr.Tab("Attribution Explorer"):
            gr.Markdown(" ## Explore a Marketing Channel ")
            gr.Markdown(
                " Select a channel to examine its lead generation, "
                " conversion and revenue contribution. "
            )
            channel_select = gr.Dropdown(
                choices=list(channels.keys()),
                value="Paid Search",
                label="Select Marketing Channel"
            )
            channel_report = gr.Markdown(
                value=explore_channel("Paid Search")
            )
            channel_select.change(
                fn=explore_channel,
                inputs=channel_select,
                outputs=channel_report
            )

        # TAB 5: AI-STYLE INSIGHTS
        with gr.Tab("Marketing Insights"):
            gr.Markdown(" ## Recommendations for Marketing Decisions ")
            insight_output = gr.Markdown(value=generate_insights())
            refresh_button = gr.Button(
                "Refresh Insights",
                variant="primary"
            )
            refresh_button.click(
                fn=generate_insights,
                inputs=[],
                outputs=insight_output
            )
            gr.Markdown(
                " Recommendations are generated from the sample data using "
                " predefined business rules. Validate them against real "
                " campaign results before making budget decisions. "
            )

    gr.Markdown(
        " ---\n "
        " **Project note:** This dashboard uses synthetic demonstration "
        " data. Channel attribution is illustrative and assigns each lead "
        " to one recorded channel; it does not track a real customer's "
        " complete multi-touch journey. "
    )

# ------------------------------------------------ --
# 8. LAUNCH THE DASHBOARD
# ------------------------------------------------ --
if __name__ == "__main__":
    demo.launch(share=True, debug=False)

/tmp/ipykernel_504/2599197882.py:340: UserWarning:

The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.



Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1cd4793ec4fcbc0466.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
import gradio as gr
import pandas as pd
import numpy as np
import plotly.express as px

# ------------------------------------------------ --
# 1. PROJECT SETTINGS
# ------------------------------------------------ --
TITLE = "Marketing-to-Revenue Attribution"
GREEN = "#24483F"
GOLD = "#C49A52"
SAGE = "#91AA9A"
DARK = "#263C34"
IVORY = "#F7F5EF"
PALE = "#E4E9E2"

np.random.seed(42)

# ------------------------------------------------ --
# 2. CREATE SAMPLE MARKETING DATA
# ------------------------------------------------ --
channels = {
    "Social Media": {
        "campaigns": ["Brand Awareness", "Product Campaign", "Retargeting"],
        "cost_per_lead": 42,
        "qualification_rate": 0.36,
        "conversion_rate": 0.19,
        "revenue": 2600,
    },
    "Email Marketing": {
        "campaigns": ["Welcome Series", "Lead Nurturing", "Product Updates"],
        "cost_per_lead": 18,
        "qualification_rate": 0.52,
        "conversion_rate": 0.31,
        "revenue": 3200,
    },
    "Paid Search": {
        "campaigns": ["High Intent Search", "Product Keywords", "Competitor Search"],
        "cost_per_lead": 65,
        "qualification_rate": 0.58,
        "conversion_rate": 0.34,
        "revenue": 3900,
    },
    "Webinars": {
        "campaigns": ["Industry Webinar", "Product Workshop", "Expert Session"],
        "cost_per_lead": 55,
        "qualification_rate": 0.64,
        "conversion_rate": 0.38,
        "revenue": 4500,
    },
    "Content Marketing": {
        "campaigns": ["Blog Campaign", "Case Study", "Downloadable Guide"],
        "cost_per_lead": 28,
        "qualification_rate": 0.45,
        "conversion_rate": 0.25,
        "revenue": 3000,
    },
}

rows = []
for i in range(1200):
    channel = np.random.choice(list(channels.keys()))
    details = channels[channel]
    campaign = np.random.choice(details["campaigns"])
    lead_score = int(np.clip(np.random.normal(65, 18), 10, 100))
    qualified_probability = np.clip(
        details["qualification_rate"] + (lead_score - 65) * 0.003,
        0.05, 0.90
    )
    qualified = np.random.random() < qualified_probability
    if qualified:
        conversion_probability = np.clip(
            details["conversion_rate"] + (lead_score - 65) * 0.002,
            0.05, 0.85
        )
        converted = np.random.random() < conversion_probability
    else:
        converted = False
    lead_cost = details["cost_per_lead"] * np.random.uniform(0.8, 1.2)
    revenue = (
        details["revenue"] * np.random.uniform(0.75, 1.25)
        if converted else 0
    )
    rows.append({
        "Lead ID": f"LEAD-{i+1:04d}",
        "Channel": channel,
        "Campaign": campaign,
        "Lead Score": lead_score,
        "Qualified": "Yes" if qualified else "No",
        "Converted": "Yes" if converted else "No",
        "Marketing Cost": round(lead_cost, 2),
        "Revenue": round(revenue, 2),
        "Days to Convert": (
            int(np.random.randint(3, 46)) if converted else None
        )
    })

df = pd.DataFrame(rows)

# ------------------------------------------------ --
# 3. PREPARE ATTRIBUTION METRICS
# ------------------------------------------------ --
def build_channel_summary(data):
    summary = data.groupby("Channel").agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Marketing_Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()
    summary["Lead-to-Qualified %"] = (
        summary["Qualified_Leads"] / summary["Leads"] * 100
    ).round(1)
    summary["Lead-to-Conversion %"] = (
        summary["Conversions"] / summary["Leads"] * 100
    ).round(1)
    summary["Cost per Lead"] = (
        summary["Marketing_Spend"] / summary["Leads"]
    ).round(2)
    summary["Revenue per Conversion"] = (
        summary["Revenue"] / summary["Conversions"].replace(0, np.nan)
    ).fillna(0).round(2)
    summary["ROAS"] = (
        summary["Revenue"] / summary["Marketing_Spend"].replace(0, np.nan)
    ).fillna(0).round(2)
    return summary

summary = build_channel_summary(df)
total_leads = len(df)
total_qualified = (df["Qualified"] == "Yes").sum()
total_conversions = (df["Converted"] == "Yes").sum()
total_spend = df["Marketing Cost"].sum()
total_revenue = df["Revenue"].sum()
overall_conversion = total_conversions / total_leads * 100
overall_roas = total_revenue / total_spend if total_spend else 0

# Perfectly Spaced HTML KPI Card Renderer
def kpi_card(label, value):
    return f"""
    <div style="background: #FFFFFF; border: 1px solid #D8DED4; border-radius: 8px; padding: 18px 12px; text-align: center; width: 100%; box-shadow: 0 1px 3px rgba(0,0,0,0.02);">
        <div style="font-size: 12px; font-weight: 700; color: #527A61; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 6px;">{label}</div>
        <div style="font-size: 24px; font-weight: 700; color: #24483F;">{value}</div>
    </div>
    """

# ------------------------------------------------ --
# 4. CREATE CHARTS
# ------------------------------------------------ --
def create_funnel():
    funnel = pd.DataFrame({
        "Stage": ["Leads Generated", "Qualified Leads", "Conversions"],
        "Count": [total_leads, total_qualified, total_conversions]
    })
    fig = px.funnel(
        funnel,
        x="Count",
        y="Stage",
        title="Marketing Funnel: From Leads to Revenue",
        color="Stage",
        color_discrete_sequence=[GREEN, SAGE, GOLD]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False
    )
    return fig

def create_revenue_chart():
    chart_data = summary.sort_values("Revenue", ascending=True)
    fig = px.bar(
        chart_data,
        x="Revenue",
        y="Channel",
        orientation="h",
        title="Revenue Attributed to Each Channel",
        color="Revenue",
        color_continuous_scale=[SAGE, GREEN],
        labels={"Revenue": "Revenue", "Channel": "Marketing Channel"}
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        coloraxis_showscale=False
    )
    return fig

def create_conversion_chart():
    chart_data = summary.sort_values("Lead-to-Conversion %", ascending=False)
    fig = px.bar(
        chart_data,
        x="Channel",
        y="Lead-to-Conversion %",
        title="Lead-to-Customer Conversion Rate by Channel",
        color="Channel",
        color_discrete_sequence=[GREEN, GOLD, SAGE, "#748A80", "#D8C9A7"]
    )
    fig.update_layout(
        paper_bgcolor=IVORY,
        plot_bgcolor=IVORY,
        font_color=DARK,
        showlegend=False,
        xaxis_title="Marketing Channel",
        yaxis_title="Conversion Rate (%)"
    )
    return fig

# ------------------------------------------------ --
# 5. INTERACTIVE FUNCTIONS
# ------------------------------------------------ --
def get_campaign_data(selected_channel):
    data = df.copy()
    if selected_channel != "All Channels":
        data = data[data["Channel"] == selected_channel]
    campaign_summary = data.groupby(["Channel", "Campaign"]).agg(
        Leads=("Lead ID", "count"),
        Qualified_Leads=("Qualified", lambda x: (x == "Yes").sum()),
        Conversions=("Converted", lambda x: (x == "Yes").sum()),
        Spend=("Marketing Cost", "sum"),
        Revenue=("Revenue", "sum")
    ).reset_index()
    campaign_summary["Conversion Rate (%)"] = (
        campaign_summary["Conversions"] / campaign_summary["Leads"] * 100
    ).round(1)
    campaign_summary["ROAS"] = (
        campaign_summary["Revenue"] /
        campaign_summary["Spend"].replace(0, np.nan)
    ).fillna(0).round(2)
    campaign_summary["Spend"] = campaign_summary["Spend"].round(2)
    campaign_summary["Revenue"] = campaign_summary["Revenue"].round(2)
    return campaign_summary

def explore_channel(selected_channel):
    channel_data = summary[summary["Channel"] == selected_channel].iloc[0]
    explanation = f"""## {selected_channel}
- **Leads generated:** {int(channel_data['Leads'])}
- **Qualified leads:** {int(channel_data['Qualified_Leads'])}
- **Conversions:** {int(channel_data['Conversions'])}
- **Lead-to-conversion rate:** {channel_data['Lead-to-Conversion %']}%
- **Marketing spend:** ${channel_data['Marketing_Spend']:,.2f}
- **Attributed revenue:** ${channel_data['Revenue']:,.2f}
- **Return on ad spend (ROAS):** {channel_data['ROAS']:.2f}x

**How to interpret this:** Compare the channel's conversion rate, revenue and ROAS together. High lead volume alone does not guarantee strong revenue."""
    return explanation

def generate_insights():
    best_roas = summary.loc[summary["ROAS"].idxmax()]
    best_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmax()]
    lowest_conversion = summary.loc[summary["Lead-to-Conversion %"].idxmin()]
    highest_spend = summary.loc[summary["Marketing_Spend"].idxmax()]
    insights = f"""# Marketing Performance Insights

### 1. Strongest revenue return
**{best_roas['Channel']}** has the highest ROAS at **{best_roas['ROAS']:.2f}x**. Review its audience, campaign message and lead quality to identify what may be worth repeating.

### 2. Best lead-to-customer conversion
**{best_conversion['Channel']}** has the highest conversion rate at **{best_conversion['Lead-to-Conversion %']}%**. Consider whether similar follow-up practices could work for other channels.

### 3. Funnel improvement opportunity
**{lowest_conversion['Channel']}** has the lowest conversion rate at **{lowest_conversion['Lead-to-Conversion %']}%**. Investigate lead quality, landing-page experience and sales follow-up before reducing its budget.

### 4. Spending review
**{highest_spend['Channel']}** has the highest marketing spend (${highest_spend['Marketing_Spend']:,.2f}). Compare that investment with its attributed revenue and ROAS before deciding whether to reallocate funds.

### Recommended next steps
- Review channels with many leads but relatively few conversions.
- Compare qualified-lead rates and conversion rates.
- Test changes to targeting, messaging and follow-up.
- Reassess performance using real campaign and sales data.

*These are data-based rules and recommendations, not predictions from a trained generative AI model.*"""
    return insights

# ------------------------------------------------ --
# 6. DASHBOARD DESIGN & CONTRAST FIXES
# ------------------------------------------------ --
css = """
.gradio-container {
    max-width: 1250px !important;
    margin: auto !important;
    background: #F7F5EF !important;
    color: #263C34 !important;
    font-family: Arial, sans-serif !important;
}
.gradio-container h2, .gradio-container h3 {
    color: #24483F !important;
}
.gradio-container label {
    color: #263C34 !important;
    font-weight: 600 !important;
}
.gradio-container button {
    border-radius: 6px !important;
}
.gradio-container input,
.gradio-container select,
.gradio-container textarea {
    background-color: #FFFFFF !important;
    color: #263C34 !important;
    -webkit-text-fill-color: #263C34 !important;
    border-color: #D8DED4 !important;
}
.gradio-container .block {
    border-color: #E0E1D8 !important;
    box-shadow: none !important;
}
"""

# ------------------------------------------------ --
# 7. BUILD THE INTERACTIVE APP
# ------------------------------------------------ --
with gr.Blocks(title=TITLE) as demo:
    gr.HTML("""
    <div style="padding: 28px 30px; border-radius: 8px; background: #24483F; color: #FFFFFF !important; margin-bottom: 22px;">
        <h1 style="color: #FFFFFF !important; font-size: 29px !important; font-weight: 600 !important; margin: 0 0 8px 0;">Marketing-to-Revenue Attribution</h1>
        <p style="color: #E4E9E2 !important; font-size: 15px !important; margin: 0;">From campaign exposure to qualified leads, conversions, and revenue.</p>
    </div>
    """)
    gr.Markdown(
        "Explore where marketing generates value, where the funnel loses "
        "potential customers, and which channels deserve closer review."
    )
    with gr.Tabs():
        # TAB 1: EXECUTIVE OVERVIEW
        with gr.Tab("Executive Overview"):
            gr.Markdown("## Business Performance at a Glance")
            with gr.Row():
                with gr.Column():
                    gr.HTML(kpi_card("Total Leads", f"{total_leads:,}"))
                with gr.Column():
                    gr.HTML(kpi_card("Qualified Leads", f"{total_qualified:,}"))
                with gr.Column():
                    gr.HTML(kpi_card("Conversions", f"{total_conversions:,}"))
            with gr.Row():
                with gr.Column():
                    gr.HTML(kpi_card("Marketing Spend ($)", f"${total_spend:,.2f}"))
                with gr.Column():
                    gr.HTML(kpi_card("Attributed Revenue ($)", f"${total_revenue:,.2f}"))
                with gr.Column():
                    gr.HTML(kpi_card("Overall ROAS (x)", f"{overall_roas:.2f}x"))
            with gr.Row():
                gr.Plot(value=create_funnel())
                gr.Plot(value=create_revenue_chart())
            gr.Markdown(
                f"**Overall lead-to-customer conversion rate:** "
                f"{overall_conversion:.1f}%"
            )

        # TAB 2: FUNNEL ANALYSIS
        with gr.Tab("Funnel Analysis"):
            gr.Markdown("## Where Are Potential Customers Lost?")
            gr.Markdown(
                "Follow the journey from lead generation to qualification "
                "and conversion. A lower count at each stage represents "
                "potential drop-off."
            )
            gr.Plot(value=create_funnel())
            gr.Plot(value=create_conversion_chart())
            funnel_table = pd.DataFrame({
                "Funnel Stage": [
                    "Leads Generated",
                    "Qualified Leads",
                    "Conversions"
                ],
                "Number of Leads": [
                    total_leads,
                    total_qualified,
                    total_conversions
                ],
                "Share of Original Leads (%)": [
                    100,
                    round(total_qualified / total_leads * 100, 1),
                    round(total_conversions / total_leads * 100, 1)
                ]
            })
            gr.Dataframe(
                value=funnel_table,
                label="Funnel Stage Summary",
                interactive=False
            )

        # TAB 3: CAMPAIGN PERFORMANCE
        with gr.Tab("Campaign Performance"):
            gr.Markdown("## Compare Individual Campaigns")
            channel_filter = gr.Dropdown(
                choices=["All Channels"] + list(channels.keys()),
                value="All Channels",
                label="Filter by Marketing Channel"
            )
            campaign_table = gr.Dataframe(
                value=get_campaign_data("All Channels"),
                label="Campaign Performance",
                interactive=False
            )
            channel_filter.change(
                fn=get_campaign_data,
                inputs=channel_filter,
                outputs=campaign_table
            )
            gr.Markdown(
                "**ROAS** means revenue attributed to a channel divided by "
                "its marketing spend. A higher ROAS is not the same as "
                "higher total profit."
            )

        # TAB 4: ATTRIBUTION EXPLORER
        with gr.Tab("Attribution Explorer"):
            gr.Markdown("## Explore a Marketing Channel")
            gr.Markdown(
                "Select a channel to examine its lead generation, "
                "conversion and revenue contribution."
            )
            channel_select = gr.Dropdown(
                choices=list(channels.keys()),
                value="Paid Search",
                label="Select Marketing Channel"
            )
            channel_report = gr.Markdown(
                value=explore_channel("Paid Search")
            )
            channel_select.change(
                fn=explore_channel,
                inputs=channel_select,
                outputs=channel_report
            )

        # TAB 5: AI-STYLE INSIGHTS
        with gr.Tab("Marketing Insights"):
            gr.Markdown("## Recommendations for Marketing Decisions")
            insight_output = gr.Markdown(value=generate_insights())
            refresh_button = gr.Button(
                "Refresh Insights",
                variant="primary"
            )
            refresh_button.click(
                fn=generate_insights,
                inputs=[],
                outputs=insight_output
            )
            gr.Markdown(
                "Recommendations are generated from the sample data using "
                "predefined business rules. Validate them against real "
                "campaign results before making budget decisions."
            )

    gr.Markdown(
        "---\n"
        "**Project note:** This dashboard uses synthetic demonstration "
        "data. Channel attribution is illustrative and assigns each lead "
        "to one recorded channel; it does not track a real customer's "
        "complete multi-touch journey."
    )

# ------------------------------------------------ --
# 8. LAUNCH THE DASHBOARD
# ------------------------------------------------ --
if __name__ == "__main__":
    demo.launch(share=True, debug=False, css=css)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7d3c915bf45792441c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
